# Datamine INDATA Process Example

This notebook demonstrates how to configure and run the **`indata`** process wrapper in `dmstudio`.

## Process Description

## INDATA

Enters free-format data (i.e. with the data items separated by commas) from the keyboard or from a system character file external to the database.

A data definition must already exist, and is read from file &**IN**. Any Datamine file may be used as a data definition. This defines the names of the fields, whether the field is numeric or alphanumeric, and if alphanumeric, the length. It also defines each fields default value, and if a field is an implicit or explicit field.

The _records_ in the &**IN** file are not used (so a block model or a prototype model copied from a block model without records would produce the same result.

The new file is written to &**OUT**. If &**IN** and &**OUT** are the same name then the new data will replace any that may exist in the old (an "in-place update"). Otherwise a new file is created, leaving the old file intact.

* **Note** (Data records may not start with the character "!". This is because the ! symbol acts as the end-of-data character. However, macro files, where the command starts with !, may be read if a blank is inserted prior to the ! symbol.):

### Input Files:

* **in** (Table):
  File containing Data Definition.
  Required=Yes

### Output Files:

* **out** (Table):
  File to be created (may be same as IN; if it is, then original data in file is
  overwritten).
  Required=Yes

### Fields:

### Parameters:

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('indata')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute indata
print("Running indata...")
dm_cmd.indata(
    in_i='t_assays',  # required
    out_o='t_indata_out',  # required
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("indata execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_indata_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")